# Day 9 · Spark SQL
## One engine, two languages

**Databricks + Snowflake · 70-Hour Programme · DataTrends.tech**

---

| Day | What you gained |
|---|---|
| 5 · 6 | DataFrames — reading and writing data |
| 7 | Transformations — `select`, `filter`, `groupBy`, joins |
| 8 | Actions, and `explain()` — reading the plan Spark builds |
| **9** | **SQL over the same DataFrames — views, the catalog, Python UDFs** |

Today adds no new engine. It adds a second way to address the one you already have.

**Free Edition note:** notebooks here are Python and SQL only. Scala and R are Paid-edition
languages, which is why every user-defined function in this session is written in Python.

## Configuration

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf
import io
import contextlib
import time

CATALOG      = "workspace"
SOURCE_TABLE = "workspace.default.upi_transactions_2026"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("USE SCHEMA default")

print(f"Source table    : {SOURCE_TABLE}")
print(f"Current catalog : {spark.catalog.currentCatalog()}")
print(f"Current schema  : {spark.catalog.currentDatabase()}")

## The table

A Unity Catalog table of UPI transactions. It is a real, registered, permanent object —
which matters for the first point of the session: SQL can already see it.

In [ ]:
df = spark.table(SOURCE_TABLE)
df.printSchema()

In [ ]:
display(df.limit(5))

---
# 1 · The same question, twice

**Industry example.** In a payments team the data engineer writes PySpark and the risk
analyst writes SQL. They query the same tables and are measured on the same numbers. If
the two languages produced different work, one of them would have to be retrained. They
do not, and neither is.

Below is one business question — value and volume by bank — expressed both ways.

In [ ]:
by_bank_api = (
    df.groupBy("bank")
      .agg(F.count("*").alias("txns"),
           F.sum("amount_inr").alias("total_inr"))
      .orderBy(F.desc("total_inr"))
)

display(by_bank_api)

In [ ]:
by_bank_sql = spark.sql(f"""
    SELECT bank,
           COUNT(*)        AS txns,
           SUM(amount_inr) AS total_inr
    FROM {SOURCE_TABLE}
    GROUP BY bank
    ORDER BY total_inr DESC
""")

display(by_bank_sql)

### What `spark.sql()` returned

Not a result. A DataFrame — the same type the API call returned, still lazy, still
unexecuted until an action asks for rows. SQL text is a *front end*. It is parsed into the
same tree of operations that the DataFrame API builds directly.

In [ ]:
print(type(by_bank_api))
print(type(by_bank_sql))

### Day 8, applied

`explain()` showed us the physical plan — the sequence of operations Spark actually runs.
Two plans are printed below, one from each version of the question.

In [ ]:
def plan_of(dataframe, mode="formatted"):
    """Capture the text explain() prints, so two plans can be compared as strings."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        dataframe.explain(mode=mode)
    return buffer.getvalue()


plan_api = plan_of(by_bank_api)
plan_sql = plan_of(by_bank_sql)

print(plan_api)

In [ ]:
print(plan_sql)

In [ ]:
import re

def structure(plan):
    """The node sequence, with the per-run expression and plan ids removed."""
    return re.findall(r"^\((\d+)\)\s+(\w+)", plan, re.M)

def normalise(plan):
    """Strip #1234 style ids, which are assigned per run and differ every time."""
    return re.sub(r"#\d+", "#", plan)

print("DataFrame API :", [name for _, name in structure(plan_api)])
print()
print("SQL           :", [name for _, name in structure(plan_sql)])
print()
print("same operator sequence :", structure(plan_api) == structure(plan_sql))
print("same plan text         :", normalise(plan_api) == normalise(plan_sql))

### Why this is the case

Both paths converge on the same **unresolved logical plan** before anything is optimised.
Catalyst then resolves names against the catalog, applies the same rewrite rules to both,
and emits one physical plan.

The practical consequence: **choose the language for the reader, not for the machine.**
Multi-step pipelines with variables and functions read better in Python. Set-oriented
business logic that an analyst must review reads better in SQL. Neither choice costs
performance, so it is free to optimise for the human.

---
# 2 · Temp views — giving SQL a name to call

SQL could address the table above because the table is registered in Unity Catalog. Most
of what you build mid-pipeline is not registered anywhere — it is an intermediate
DataFrame that exists only as a plan inside your session.

**Industry example.** A reconciliation job filters a day's transactions to the disputed
ones, then runs nine different SQL checks against that filtered set. The filtered set is
not worth storing — it is rebuilt every run — but it does need a name.

In [ ]:
high_value = (
    df.filter(F.col("amount_inr") > 10000)
      .select("txn_id", "txn_time", "bank", "city", "amount_inr", "status")
)

print("high_value is a:", type(high_value).__name__)
print("can SQL address the name 'high_value'? ", spark.catalog.tableExists("high_value"))

In [ ]:
high_value.createOrReplaceTempView("high_value")

print("can SQL address the name 'high_value'? ", spark.catalog.tableExists("high_value"))

### What creating the view cost

A view is a **name bound to a plan**. No rows were read, no memory was filled, nothing was
copied. The work still happens later, when an action asks for it.

In [ ]:
display(spark.sql("""
    SELECT city,
           COUNT(*)        AS txns,
           SUM(amount_inr) AS total_inr
    FROM high_value
    GROUP BY city
    ORDER BY total_inr DESC
    LIMIT 5
"""))

### The pattern this unlocks

Python for the parts that need control flow, SQL for the parts that need to be read by
someone else — mixed freely in one notebook, over one dataset, with no export step
between them.

In [ ]:
%sql
SELECT bank,
       status,
       COUNT(*) AS txns
FROM high_value
GROUP BY bank, status
ORDER BY bank, status

---
# 3 · Scope — how long a name lives

| | Visible to | Survives notebook detach | Registered in Unity Catalog |
|---|---|---|---|
| Temp view | this session only | no | no |
| Global temp view | any session on the same cluster | no | no |
| Table (Day 10) | everyone with permission | yes | yes |

A global temp view lives in a reserved database called `global_temp` and **must be
qualified with it**. Forgetting the qualifier is the commonest error with the feature.

That middle row describes a **cluster**. Serverless gives you no cluster of your own —
so the scope a global temp view depends on does not exist here. Rather than give you a
scope that quietly means nothing, the platform refuses the operation outright.

In [ ]:
try:
    high_value.createOrReplaceGlobalTempView("high_value_global")
    display(spark.sql("SELECT COUNT(*) AS rows_visible FROM global_temp.high_value_global"))
    GLOBAL_TEMP_SUPPORTED = True
except Exception as e:
    GLOBAL_TEMP_SUPPORTED = False
    print(type(e).__name__)
    print(str(e).split("\n")[0])

### What the platform enforces

On classic compute this call succeeds, the view lands in `global_temp`, and it answers
only to `global_temp.high_value_global` — never to the bare name.

Both facts are worth carrying: the qualifier rule for interviews and for any classic
cluster you meet, and the refusal above as evidence that scope on this platform is
enforced rather than merely documented.

In [ ]:
print(f"global temp view created : {GLOBAL_TEMP_SUPPORTED}")

for name in ("high_value", "global_temp.high_value_global"):
    try:
        print(f"{name:32} resolvable: {spark.catalog.tableExists(name)}")
    except Exception as e:
        print(f"{name:32} resolvable: no  ({type(e).__name__})")

---
# 4 · The catalog — what exists right now

**Industry example.** A scheduled job creates six temp views. It fails on the fourth. The
on-call engineer needs to know which names were built before it died — without rerunning
anything.

In [ ]:
display(spark.sql("SHOW VIEWS"))

In [ ]:
try:
    display(spark.sql("SHOW VIEWS IN global_temp"))
except Exception as e:
    print("global_temp is not available on this compute")
    print(type(e).__name__)

In [ ]:
print("temp view registered   :", spark.catalog.tableExists("high_value"))
print("source table exists    :", spark.catalog.tableExists(SOURCE_TABLE))

### Names are a resource — release them

Views cost nothing to store, but a stale name pointing at last hour's logic is a real
source of wrong answers in long-running notebooks.

In [ ]:
spark.catalog.dropTempView("high_value")

try:
    spark.catalog.dropGlobalTempView("high_value_global")
except Exception:
    pass

print("temp view registered   :", spark.catalog.tableExists("high_value"))

---
# 5 · Python UDFs — when nothing built in will do

**Industry example.** Transaction identifiers are being shared with an external
reconciliation vendor. Contract says the middle of every identifier must be obscured, the
first four characters kept for routing and the last two kept for support lookups.

No single built-in function implements that rule, so we write one.

In [ ]:
@udf(returnType=StringType())
def mask_txn(txn_id):
    if txn_id is None or len(txn_id) < 6:
        return txn_id
    return txn_id[:4] + "*" * (len(txn_id) - 6) + txn_id[-2:]


display(
    df.select("txn_id", mask_txn(F.col("txn_id")).alias("masked_txn_id")).limit(5)
)

### The same function, available to SQL

Registering it puts the function in the session's function registry, where SQL text can
resolve it — the same move `createOrReplaceTempView` made for data.

In [ ]:
spark.udf.register("mask_txn_sql", mask_txn)

display(spark.sql(f"""
    SELECT txn_id,
           mask_txn_sql(txn_id) AS masked_txn_id
    FROM {SOURCE_TABLE}
    LIMIT 5
"""))

### What a UDF costs

Spark's engine runs on the JVM. Your function is Python. Every row a Python UDF touches
must leave the JVM, be deserialised into Python, be processed by the interpreter, and be
serialised back.

That boundary is visible in the plan.

In [ ]:
via_udf = df.select(mask_txn(F.col("txn_id")).alias("m"))

via_udf.explain()

### The same rule expressed with built-ins

A built-in stays inside the engine, is visible to Catalyst, and can be compiled into the
generated code for the stage.

In [ ]:
MASK_PATTERN = "(?<=.{4}).(?=.{2})"

via_builtin = df.select(
    F.regexp_replace(F.col("txn_id"), MASK_PATTERN, "*").alias("m")
)

via_builtin.explain()

### Do the two agree?

A rewrite that is faster and wrong is not a rewrite. Before trusting the built-in version
we check it against the function whose behaviour we defined.

In [ ]:
comparison = df.select(
    F.col("txn_id"),
    mask_txn(F.col("txn_id")).alias("via_udf"),
    F.regexp_replace(F.col("txn_id"), MASK_PATTERN, "*").alias("via_builtin"),
)

disagreements = comparison.filter(F.col("via_udf") != F.col("via_builtin")).count()

print(f"rows compared     : {df.count():,}")
print(f"rows disagreeing  : {disagreements:,}")

### Cost of the boundary, measured

Both versions are forced to actually run by aggregating over their output — a projection
that nothing consumes would simply be optimised away.

In [ ]:
_t0 = time.time()
via_udf.agg(F.sum(F.length("m"))).collect()
udf_secs = round(time.time() - _t0, 2)

_t0 = time.time()
via_builtin.agg(F.sum(F.length("m"))).collect()
builtin_secs = round(time.time() - _t0, 2)

print(f"Python UDF : {udf_secs}s")
print(f"Built-in   : {builtin_secs}s")

### The rule to leave with

1. Reach for a **built-in** first. Catalyst can see it, optimise it and compile it.
2. Reach for a **Python UDF** when the rule genuinely has no built-in equivalent — bespoke
   business logic, an external library, a lookup no SQL expression can express.
3. If a UDF is unavoidable and the data is large, a **Pandas (vectorised) UDF** moves rows
   across the boundary in Arrow batches instead of one at a time.

A UDF is not forbidden. It is a decision, and the plan tells you what it cost.

---
# 6 · What today did not give you

Everything named in this notebook — `high_value`, `high_value_global`, `mask_txn_sql` —
exists only until this session ends. Detach the notebook and every one of them is gone.
The UPI table survives, because somebody made it a **table**.

That is Day 10: Delta Lake — storage that has a schema, a transaction log and a history,
and that is still there tomorrow morning.

---
## Your turn — 25 minutes

Work in your own workspace on the built-in `samples.tpch` catalog. Nothing to download.

```
orders : o_orderkey · o_custkey · o_orderstatus · o_totalprice · o_orderdate
         o_orderpriority · o_clerk · o_shippriority · o_comment
```

**1 — Two languages, one plan.**
Count orders and total `o_totalprice` by `o_orderpriority`, once with the DataFrame API and
once with `spark.sql`. Compare the two plans.

**2 — Name something SQL cannot see.**
Build a DataFrame of orders with `o_totalprice > 300000`. Register it as a temp view called
`big_orders`. Using SQL on that view, find the top 5 `o_clerk` values by number of orders.

**3 — Prove the scope.**
Show that `big_orders` is temporary and does not appear in Unity Catalog. Then create a
global temp view from the same DataFrame and query it correctly.

**4 — A rule with no built-in.**
Write a Python UDF `order_band(total)` returning `SMALL` under 50,000, `MEDIUM` under
200,000, otherwise `LARGE`. Register it for SQL and produce a count of orders per band.

**5 — Stretch.**
Rewrite task 4 as a SQL `CASE WHEN`. Compare `explain()` output against the UDF version and
write one sentence describing the difference you see.

---
## Extension · Unity Catalog Python UDFs

A `spark.udf.register` function dies with the session. Unity Catalog can hold a Python
function as a permanent, governed object that analysts can call from SQL without a
notebook — the natural production form of section 5.

This is a recent capability and its availability should be confirmed in the current
workspace before it is relied on.

In [ ]:
spark.sql("""
    CREATE OR REPLACE FUNCTION workspace.default.mask_txn_uc(txn_id STRING)
    RETURNS STRING
    LANGUAGE PYTHON
    AS $$
        if txn_id is None or len(txn_id) < 6:
            return txn_id
        return txn_id[:4] + "*" * (len(txn_id) - 6) + txn_id[-2:]
    $$
""")

display(spark.sql(f"""
    SELECT txn_id,
           workspace.default.mask_txn_uc(txn_id) AS masked_txn_id
    FROM {SOURCE_TABLE}
    LIMIT 5
"""))

## Reset

Removes every object this notebook created. The source table is untouched.

In [ ]:
for _v in ("high_value",):
    spark.sql(f"DROP VIEW IF EXISTS {_v}")

try:
    spark.sql("DROP VIEW IF EXISTS global_temp.high_value_global")
except Exception:
    pass

spark.sql("DROP FUNCTION IF EXISTS workspace.default.mask_txn_uc")

print("session objects removed")